## 1. Imports & Environment Setup 

In [11]:
# Initial Imports
from pathlib import Path
import json
import pandas as pd
import sys, os
from dotenv import load_dotenv
from mozzarellm.utils.io import load_table
from mozzarellm.utils.screen_context_utils import load_screen_context_json
from mozzarellm.pipeline.bundle_builder import (
    build_evidence_bundles,
    get_or_append_stable_accession,
)
from mozzarellm.clients.llm_api_clients import create_client
from mozzarellm.utils.prompt_factory import (
    make_cluster_analysis_system_prompt,
    make_single_cluster_analysis_user_prompt,
)
from mozzarellm.utils.cluster_utils import build_cluster_id_to_bundle_path

print(sys.path)

['C:\\Users\\alexa\\OneDrive - Massachusetts Institute of Technology\\Cheesman Lab\\WORKSPACE\\mozzarellm', 'C:\\Users\\alexa\\AppData\\Local\\Python\\pythoncore-3.12-64\\python312.zip', 'C:\\Users\\alexa\\AppData\\Local\\Python\\pythoncore-3.12-64\\DLLs', 'C:\\Users\\alexa\\AppData\\Local\\Python\\pythoncore-3.12-64\\Lib', 'C:\\Users\\alexa\\AppData\\Local\\Python\\pythoncore-3.12-64', 'c:\\Users\\alexa\\OneDrive - Massachusetts Institute of Technology\\Cheesman Lab\\WORKSPACE\\mozzarellm\\.mozzvenv', '', 'c:\\Users\\alexa\\OneDrive - Massachusetts Institute of Technology\\Cheesman Lab\\WORKSPACE\\mozzarellm\\.mozzvenv\\Lib\\site-packages', '__editable__.mozzarellm-0.2.0.finder.__path_hook__']


In [12]:
def _find_repo_root(start_dir: Path) -> Path:
    start_dir = start_dir.resolve()
    for p in [start_dir] + list(start_dir.parents):
        if (p / "pyproject.toml").exists():
            return p
    raise RuntimeError(f"Could not find pyproject.toml starting from {start_dir}")


# Use repo root for sys.path and for locating .env, regardless of notebook working directory (less brittle).
repo_root = _find_repo_root(Path(os.getcwd()))
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

# Load secrets/config from .env at the repo root (if present)
print("Does .env exist in repo root?", (repo_root / ".env").exists())
load_dotenv(dotenv_path=repo_root / ".env")
print("Repo root:", repo_root)
print("Current working directory:", os.getcwd())

Does .env exist in repo root? True
Repo root: C:\Users\alexa\OneDrive - Massachusetts Institute of Technology\Cheesman Lab\WORKSPACE\mozzarellm
Current working directory: c:\Users\alexa\OneDrive - Massachusetts Institute of Technology\Cheesman Lab\WORKSPACE\mozzarellm\interface


## 2. Screen Context Configuration

Complete the `screen_context.json` file with the relevant information for your screen. Feel free to adjust the template as needed beyond the required fields. The top level must be a dictionary, and all fields must be strings. Keep the file in the same directory as the notebook. This cell will check for the file and raise an error if it is not found or not well formatted. 

In [13]:
# set screen name
SCREEN_NAME = "test"
SCREEN_CONTEXT_PATH = Path("screen_context.json")  # or change to your custom filename.
screen_ctx = load_screen_context_json(SCREEN_CONTEXT_PATH)
# should throw error if file doesn't exist or if you forgot to remove the TODO!

print("Loaded screen_context.json:", isinstance(screen_ctx, dict))
keys = list(screen_ctx.keys())
print(f"Top-level keys ({len(keys)}): {keys}")

Loaded screen_context.json: True
Top-level keys (10): ['assay_type', 'target_phenotype', 'organism', 'cell_line_or_system', 'perturbation', 'readout', 'clustering', 'controls', 'provenance', 'notes']


## 3. Load & Standardize Cluster Table 
### **3.1** Initial Load
Set parameters and check that the original cluster table loads.


In [14]:
#### 3.1.1 Point this to your cluster/gene table (CSV/TSV/TXT/XLSX) ####
default_cluster_table = (
    repo_root / "examples" / "ops" / "funk_2022.csv"
)  # NOTE: using funk_2022.csv for development
CLUSTER_TABLE_PATH = Path(default_cluster_table)

#### 3.1.2 Optional: ####
sep = None
# Useful for .txt files. If the table loads as one big column (or in an otherwise wonky fashion), the delimiter is likely mismatched.
# By default pandas ASSUMES the delimiter (value separator) based on file extension
# (typically .csv → ',', .tsv → '\t'). Common overrides are ',', '|', '\t', or ';'.

sheet_name = None
# For Excel only (e.g. 0 or 'Sheet1'); ignored for CSV/TSV
#########################

cluster_df = load_table(CLUSTER_TABLE_PATH, sep=sep, sheet_name=sheet_name)
print("Loaded:", str(CLUSTER_TABLE_PATH))
print("Shape:", cluster_df.shape)
print("Columns:\n", list(cluster_df.columns))
display(cluster_df.head())

Loaded: C:\Users\alexa\OneDrive - Massachusetts Institute of Technology\Cheesman Lab\WORKSPACE\mozzarellm\examples\ops\funk_2022.csv
Shape: (164, 5)
Columns:
 ['gene_symbol', 'cluster', 'up_features', 'down_features', 'phenotypic_strength']


,gene_symbol,cluster,up_features,down_features,phenotypic_strength
0,AATF,21,"interphase_cell_correlation_dapi_tubulin,inter...","interphase_nucleus_area,interphase_cell_area,i...",669/5299
1,ABT1,21,"interphase_nucleus_gh2ax_mean,interphase_nucle...","interphase_nucleus_area,interphase_cell_area,i...",560/5299
2,BMS1,21,"interphase_nucleus_dapi_mean,interphase_cell_c...","interphase_nucleus_area,interphase_cell_area,i...",446/5299
3,BYSL,21,"interphase_nucleus_dapi_mean,interphase_cell_c...","interphase_nucleus_area,interphase_cell_area,i...",524/5299
4,C1orf131,21,"interphase_nucleus_dapi_mean,interphase_cell_c...","interphase_nucleus_area,interphase_cell_area,i...",343/5299


### **3.2** Gene to stable accession ID mapping

If you do not have stable accession IDs in your cluster table, we will make a best effort attempt to assign them based on your provided gene symbols (NOTE: the `get_accession_from_gene_symbol` method is available for review in `clients/uniprot_api_client.py`).

If you have a gene to stable accession IDs maping available separately from your cluster table, you can provide this table, and we will append the accession IDs to your cluster table. 

A csv copy of the assigned or appended gene to stable accession mapping will be provided for review in the `interface/output/` directory.

In [15]:
# set cluster id column name or keep default
CLUSTER_ID_COLUMN = "cluster"
# set gene column name or keep default
GENE_COLUMN = "gene_symbol"

# If you have a stable accession column already in your cluster table, set it here.
# If no stable accession column, leave as None and we'll handle it for you.
STABLE_ACCESSION_COLUMN = None  # or the name of your stable accession column

# List of per-gene feature columns that would be useful to determine overlaps for within a cluster.
FEATURE_COLUMNS = [
    "up_features",
    "down_features",
]

# Set organism id or keep default
ORGANISM_ID = 9606  # human

if STABLE_ACCESSION_COLUMN is None:
    ACC_CLUSTER_DF = get_or_append_stable_accession(
        cluster_df=cluster_df,
        # IF ADDING A SEPERATE TABLE, SET THESE
        accession_table=None,  # or Path("path/to/accession_table.csv")
        accession_col=None,  # or the name of your stable accession column in the seperate table
        accession_table_sep=None,  # or the delimiter for the seperate table (use if you encounter encoding issues)
        accession_table_sheetname=None,  # or the sheet name for the seperate table if it's an excel file
        organism_id=ORGANISM_ID,
        warn_on_fallback=False,  # default is to ignore warnings for unreviewed entries
    )

# NOTE: Currently, running this cell will overwrite the last .csv output.
# TODO: (later) add option to prevent overwriting with a timestamp in filename. [just a hassle during iteration]
ACC_CLUSTER_DF

,gene_symbol,cluster,up_features,down_features,phenotypic_strength,accession
0,AATF,21,"interphase_cell_correlation_dapi_tubulin,inter...","interphase_nucleus_area,interphase_cell_area,i...",669/5299,Q9NY61
1,ABT1,21,"interphase_nucleus_gh2ax_mean,interphase_nucle...","interphase_nucleus_area,interphase_cell_area,i...",560/5299,Q9ULW3
2,BMS1,21,"interphase_nucleus_dapi_mean,interphase_cell_c...","interphase_nucleus_area,interphase_cell_area,i...",446/5299,Q14692
3,BYSL,21,"interphase_nucleus_dapi_mean,interphase_cell_c...","interphase_nucleus_area,interphase_cell_area,i...",524/5299,Q13895
4,C1orf131,21,"interphase_nucleus_dapi_mean,interphase_cell_c...","interphase_nucleus_area,interphase_cell_area,i...",343/5299,Q8NDD1
...,...,...,...,...,...,...
159,nontargeting_5_2,99,NaN,NaN,5203/5299,NON_TARGETING_CONTROL
160,nontargeting_61_0,99,NaN,NaN,5038/5299,NON_TARGETING_CONTROL
161,nontargeting_6_1,99,NaN,interphase_cell_phalloidin_mean,5237/5299,NON_TARGETING_CONTROL
162,nontargeting_8_2,99,NaN,NaN,5271/5299,NON_TARGETING_CONTROL


## 4. Create Per-Cluster Evidence Bundles

Bundles will be saved to the `interface/output` directory. Additionally, each chunk has retrieved annotations and evidence logged in a corresponding csv file within the same directory. These can be accessed via File Explorer and viewed in Excel or Google Sheets.

In [16]:
build_evidence_bundles(
    screen_name=SCREEN_NAME,
    acc_cluster_df=ACC_CLUSTER_DF,  # cluster table must have accession numbers
    gene_column=GENE_COLUMN,
    cluster_id_column=CLUSTER_ID_COLUMN,
    stable_accession_col=STABLE_ACCESSION_COLUMN or "accession",
    feature_columns=FEATURE_COLUMNS,
    use_retrieval=False,  # false for now; true for future use
    knowledge_dir=None,
    top_k=5,
)

Using gene column: gene_symbol
Processing 7 clusters
Processing cluster 21
Saved bundle to output\test_evidence_bundles\test__cluster_21__bundle.json
Processing cluster 37
Saved bundle to output\test_evidence_bundles\test__cluster_37__bundle.json
Processing cluster 121
Saved bundle to output\test_evidence_bundles\test__cluster_121__bundle.json
Processing cluster 149
Saved bundle to output\test_evidence_bundles\test__cluster_149__bundle.json
Processing cluster 167
Saved bundle to output\test_evidence_bundles\test__cluster_167__bundle.json
Processing cluster 197
Saved bundle to output\test_evidence_bundles\test__cluster_197__bundle.json
Processing cluster 99
Saved bundle to output\test_evidence_bundles\test__cluster_99__bundle.json


In [17]:
# build cluster ID to evidence bundle path mapping
BUNDLES_DIR = Path(
    f"output/{SCREEN_NAME}_evidence_bundles"
)  # or adjust this path based on the output dir above
CLUSTER_ID_TO_BUNDLE_PATH_MAP = build_cluster_id_to_bundle_path(
    BUNDLES_DIR, screen_name=SCREEN_NAME
)
CLUSTER_ID_TO_BUNDLE_PATH_MAP

{'121': WindowsPath('output/test_evidence_bundles/test__cluster_121__bundle.json'),
 '149': WindowsPath('output/test_evidence_bundles/test__cluster_149__bundle.json'),
 '167': WindowsPath('output/test_evidence_bundles/test__cluster_167__bundle.json'),
 '197': WindowsPath('output/test_evidence_bundles/test__cluster_197__bundle.json'),
 '21': WindowsPath('output/test_evidence_bundles/test__cluster_21__bundle.json'),
 '37': WindowsPath('output/test_evidence_bundles/test__cluster_37__bundle.json'),
 '99': WindowsPath('output/test_evidence_bundles/test__cluster_99__bundle.json')}

## 5. LLM Analysis (Initial Hypothesis Formation)

In [18]:
SYSTEM_PROMPT = make_cluster_analysis_system_prompt(
    screen_context_path=Path("screen_context.json"),
    template_path=None,  # or your custom .md or .txt file to override the default template
    template_string=None,
)
print(SYSTEM_PROMPT)
# NOTE: we will be appending ""\n\n The following experimental context is provided: " + SCREEN_CONTEXT + "\n\n"" to any custom template you provide


MISSION: Functional genomics experiments cluster genes by phenotypic similarity. Your goal is to:
1. Identify the dominant biological pathway that explains why these genes cluster together
2. Classify ALL genes relative to this pathway (ESTABLISHED / UNCHARACTERIZED / NOVEL_ROLE)
3. Prioritize understudied genes (UNCHARACTERIZED and NOVEL_ROLE) for follow-up experiments

The pathway is not the end goal - it's the lens for discovering which genes merit investigation.


{"assay_type": "e.g. Optical pooled screen (OPS) morphology screen", "target_phenotype": "e.g. resistance to drug X", "organism": "e.g. Homo sapiens", "cell_line_or_system": "e.g. HeLa", "perturbation": {"type": "e.g. CRISPR KO / CRISPRi / transposon mutagenesis", "library_or_reagent": "e.g. genome-wide sgRNA library"}, "readout": {"measurement": "e.g. morphology embedding / viability / reporter", "instrument_or_platform": "e.g. Cell Painting + imaging", "primary_metric": "e.g. z-scored feature vector"}, "clustering": {"

In [25]:
# FOR SINGLE CLUSTER QUERY
# Pick a cluster ID (string keys)
CLUSTER_ID = "121"  # change me
USER_PROMPT = make_single_cluster_analysis_user_prompt(
    CLUSTER_ID, "test", CLUSTER_ID_TO_BUNDLE_PATH_MAP
)
print(
    json.dumps(
        json.loads(Path(CLUSTER_ID_TO_BUNDLE_PATH_MAP[str(CLUSTER_ID)]).read_text()), indent=2
    )
)

{
  "screen_name": "test",
  "cluster_id": "121",
  "cluster_genes": [
    {
      "gene_symbol": "CCDC174",
      "up_features": "interphase_nucleus_area,interphase_cell_dapi_mass_displacement,interphase_cell_correlation_dapi_tubulin,interphase_nucleus_eccentricity,interphase_cell_area",
      "down_features": "interphase_nucleus_form_factor,interphase_nucleus_dapi_mean,interphase_nucleus_correlation_dapi_gh2ax",
      "phenotypic_strength": "2240/5299",
      "UniProt_functional_annotation": "Probably involved in neuronal development",
      "accession": "Q6PII3"
    },
    {
      "gene_symbol": "DDA1",
      "up_features": "interphase_cell_area,interphase_cell_correlation_dapi_tubulin,interphase_nucleus_area",
      "down_features": "interphase_nucleus_correlation_dapi_gh2ax,interphase_nucleus_dapi_mean,interphase_nucleus_form_factor,interphase_nucleus_solidity",
      "phenotypic_strength": "2006/5299",
      "UniProt_functional_annotation": "Functions as a component of numerous d

In [20]:
# init client
LLM_client = create_client(
    model="claude-3-7-sonnet-20250219",  # or your preferred model
    api_key=os.getenv("ANTHROPIC_API_KEY"),  # or your preferred API key
)

############################ SINGLE CLUSTER QUERY ################################
response_text, error_msg = LLM_client.query(
    max_retries=3,
    batch=False,
    system_prompt=SYSTEM_PROMPT,
    user_prompt=USER_PROMPT,
)

if error_msg:
    raise RuntimeError(error_msg)

# Display in notebook
print(response_text)

# Save response for later inspection
out_path = Path(
    f"output/{SCREEN_NAME}_LLM_analysis/{CLUSTER_ID}_single_cluster_analysis_response.txt"
)
out_path.parent.mkdir(parents=True, exist_ok=True)
out_path.write_text(response_text, encoding="utf-8")
print(f"Saved response to: {out_path}")

################################## BATCH QUERY ##################################
# LLM_client.query(
#     max_retries=3,
#     batch=True,
#     screen_name="test",
#     system_prompt=SYSTEM_PROMPT,
#     cluster_to_prompt_map=CLUSTER_ID_TO_BUNDLE_PATH_MAP,
# )

{
  "cluster_id": "21",
  "dominant_process": "Ribosome biogenesis and pre-rRNA processing",
  "pathway_confidence": "High",
  "established_genes": [
    "AATF", "BMS1", "BYSL", "C1orf131", "DDX18", "DDX47", "DDX52", "DIMT1", 
    "EIF3D", "EIF3E", "EIF3F", "EIF3M", "EIF4A1", "NGDN", "NOP14", "PDCD11", 
    "RAN", "RBIS", "RIOK2", "RRP36", "RRP7A", "TRMT112", "TSR1", "UTP14A", "UTP20"
  ],
  "uncharacterized_genes": [
    {
      "gene": "NOC4L",
      "priority": 7,
      "rationale": "NOC4L has minimal functional annotation in UniProt, suggesting it's largely uncharacterized. However, its name (Nucleolar Complex Associated 4 Homolog) and its clustering with ribosome biogenesis genes strongly suggest a role in this pathway. The lack of detailed functional studies makes it a promising candidate for further investigation."
    },
    {
      "gene": "NOL10",
      "priority": 6,
      "rationale": "NOL10 (Nucleolar Protein 10) has no functional annotation in UniProt despite its name sug